# DFD Data Flow Diagrams
**Modules 1-2 · Beginner** · part 1 of 8 of *Building Automated Data Pipelines with Spark, dbt and Airflow* (UK · USA · Mauritius job-market aligned)

Each notebook is **self-contained**: run top to bottom. Needs Python 3.9+ and Java 8/11/17 for PySpark (not needed for 01, 05, 08).

**Series:**
- `01_DFD_Data_Flow_Diagrams.ipynb`: Modules 1-2 · Beginner
- `02_Modular_Pipeline.ipynb`: Modules 3-4 · Beginner to Intermediate
- `03_Connectors_and_Integration.ipynb`: Modules 5-6 · Intermediate
- `04_SCD2_Spark_and_dbt.ipynb`: Modules 7-8 · Intermediate to Advanced
- `05_Airflow_Workflow_Design.ipynb`: Modules 9-10 · Advanced
- `06_Capstone_End_to_End.ipynb`: Module 11 · Advanced
- `07_Advanced_Performance_Testing_CICD.ipynb`: Beyond the course · Advanced
- `08_Career_UK_USA_Mauritius.ipynb`: Job-market alignment · All levels

## Setup

In [ ]:
import os, sys, json, time, shutil, logging, sqlite3, ast
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE = Path("pipeline_lab"); BASE.mkdir(exist_ok=True)

# PART 1: Data Flow Diagrams (Modules 1–2) · Beginner

**Why it matters:** senior hires are judged on whether they can *design before coding*. DFDs are how you communicate pipelines to analysts, architects, auditors and (in UK/EU finance) regulators asking about data lineage.

**Four building blocks**
- **External entity**: source or consumer outside the system (CRM, BI tool)
- **Process**: transforms data (Ingest, Clean, Integrate)
- **Data store**: where data rests (raw zone, warehouse)
- **Data flow**: labelled arrow showing what moves

**Rules:** every process has at least one input and one output; data never flows store→store directly; label every flow with a noun phrase.

We draw a Context diagram (Level 0 as one bubble) and a Level-1 decomposition.

In [ ]:
def draw_dfd(nodes, flows, title, figsize=(13, 5)):
    kinds = {"ext": dict(boxstyle="square,pad=0.5", fc="#E8F0FE", ec="#1a73e8"),
             "proc": dict(boxstyle="round,pad=0.6", fc="#FFF4E5", ec="#e8710a"),
             "store": dict(boxstyle="round4,pad=0.5", fc="#E6F4EA", ec="#188038")}
    fig, ax = plt.subplots(figsize=figsize); ax.set_axis_off(); ax.set_title(title, fontsize=13, weight="bold")
    xs = [v[0] for v in nodes.values()]; ys = [v[1] for v in nodes.values()]
    ax.set_xlim(min(xs) - 1.5, max(xs) + 1.5); ax.set_ylim(min(ys) - 1, max(ys) + 1)
    for name, (x, y, k) in nodes.items():
        ax.text(x, y, name, ha="center", va="center", fontsize=9, bbox=dict(**kinds[k]), zorder=3)
    for src, dst, label in flows:
        (x1, y1, _), (x2, y2, _) = nodes[src], nodes[dst]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), zorder=1,
                    arrowprops=dict(arrowstyle="->", lw=1.3, shrinkA=32, shrinkB=32, color="#555"))
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.12, label, fontsize=7.5, ha="center", color="#333",
                bbox=dict(fc="white", ec="none", pad=0.5), zorder=2)
    plt.tight_layout(); plt.show()

# Context diagram
draw_dfd(
    nodes={"CRM database": (0, 1, "ext"), "Orders CSV": (0, 0, "ext"), "Web events JSON": (0, -1, "ext"),
           "0. Customer-360\nPipeline": (4, 0, "proc"),
           "BI / Analysts": (8, 0.7, "ext"), "Data Science": (8, -0.7, "ext")},
    flows=[("CRM database", "0. Customer-360\nPipeline", "customer records"),
           ("Orders CSV", "0. Customer-360\nPipeline", "order lines"),
           ("Web events JSON", "0. Customer-360\nPipeline", "click events"),
           ("0. Customer-360\nPipeline", "BI / Analysts", "customer_360 mart"),
           ("0. Customer-360\nPipeline", "Data Science", "feature tables")],
    title="Context diagram")

# Level-1 diagram
draw_dfd(
    nodes={"Sources": (0, 0, "ext"),
           "1. Ingest": (2.5, 0, "proc"), "D1 Raw zone": (2.5, -1.6, "store"),
           "2. Clean &\nvalidate": (5, 0, "proc"), "D2 Quarantine": (5, 1.6, "store"),
           "3. Integrate": (7.5, 0, "proc"),
           "4. SCD2\nhistory": (10, 0, "proc"), "D3 Warehouse\n(dim + marts)": (10, -1.6, "store"),
           "BI tools": (12.5, 0, "ext")},
    flows=[("Sources", "1. Ingest", "extracts"),
           ("1. Ingest", "D1 Raw zone", "raw files"),
           ("1. Ingest", "2. Clean &\nvalidate", "raw records"),
           ("2. Clean &\nvalidate", "D2 Quarantine", "rejects"),
           ("2. Clean &\nvalidate", "3. Integrate", "valid records"),
           ("3. Integrate", "4. SCD2\nhistory", "unified entities"),
           ("4. SCD2\nhistory", "D3 Warehouse\n(dim + marts)", "versions"),
           ("D3 Warehouse\n(dim + marts)", "BI tools", "marts")],
    title="Level-1 DFD (decomposition of process 0)", figsize=(15, 5))

**Exercise 1.1:** add a process "5. Publish marts" and a data store "D4 Metrics" that receives run statistics. Which flow feeds D4?

**Interview angle:** "Walk me through how you would design a pipeline before writing code." Answer with sources → ingestion → zones (raw/curated/serving) → quality gates → history handling → orchestration → monitoring.